In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd
from pathlib import Path

PREFIX = "Represent this sentence for searching relevant passages: "

# if __name__ == "__main__":
cases_path = Path("./data/v2/cases.dev.jsonl")
evidence_path = Path("./data/v2/evidence.jsonl")
cases_df = pd.read_json(cases_path, lines=True)
evidence_df = pd.read_json(evidence_path, lines=True)

In [9]:
cases_df

,id,split,group,pair_id,variant,task,report,question,expected_behavior,required_missing_fields,expected_action,gold_evidence_ids,gold_evidence_groups,gold_source_ids,forbidden_claims,annotation
0,CT2-001,dev,screening_framework,screening_framework,A,guideline_selection,Baseline low-dose chest CT within an organized...,"Which reporting framework should I consult, an...",选择 Lung-RADS；关键是报告明确为肺癌筛查 CT。当前只问框架选择，结节尺寸缺失不阻...,[],answer_with_evidence_and_limits,[uspstf-screening-2021::b0020],"[[uspstf-screening-2021::b0020, acr-lungrads-s...",[uspstf-screening-2021],"[不能依据筛查资格声明直接给本例结节分类, 不能把结节大小设为仅选择框架时的必答前提]","{'origin': 'assistant_generated_synthetic', 's..."
1,CT2-002,dev,screening_framework,screening_framework,B,guideline_selection,Annual screening LDCT. A 67-year-old adult has...,For interpreting this completed screening exam...,Lung-RADS 与已明确的筛查影像解释场景匹配。将筛查资格核对与已完成检查的报告框架选择...,[],answer_with_evidence_and_limits,[uspstf-screening-2021::b0020],"[[uspstf-screening-2021::b0020, acr-lungrads-s...",[uspstf-screening-2021],"[不能依据筛查资格声明直接给本例结节分类, 不能把结节大小设为仅选择框架时的必答前提]","{'origin': 'assistant_generated_synthetic', 's..."
2,CT2-003,dev,missing_examination_purpose,missing_examination_purpose,A,clarification,Chest CT: a 63-year-old adult with a single 7 ...,Which guideline pathway applies to this nodule...,先追问检查是否属于肺癌筛查项目，还是因其他原因检查时偶发发现。现有材料分别描述筛查与偶发场景...,[examination_purpose],clarify,"[uspstf-screening-2021::b0020, acr-incidental-...","[[uspstf-screening-2021::b0020], [acr-incident...","[acr-incidental-2023, uspstf-screening-2021]",[],"{'origin': 'assistant_generated_synthetic', 's..."
3,CT2-004,dev,missing_examination_purpose,missing_examination_purpose,B,clarification,A 58-year-old adult has a 9 mm solid nodule on...,Can you select a screening versus incidental-n...,不能唯一分流；具体询问检查目的／筛查项目身份，保留两类来源的适用前提。无需先追问与该层分流无...,[examination_purpose],clarify,"[uspstf-screening-2021::b0020, acr-incidental-...","[[uspstf-screening-2021::b0020], [acr-incident...","[acr-incidental-2023, uspstf-screening-2021]",[],"{'origin': 'assistant_generated_synthetic', 's..."
4,CT2-005,dev,screening_age_condition,screening_age_condition,A,eligibility_check,An asymptomatic 49-year-old adult currently sm...,Does this person meet the age and smoking cond...,吸烟暴露满足所引条款，但 49 岁低于该声明 50 岁的下限。只判断这一声明的适用条件，不据...,[],answer_with_evidence_and_limits,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",[uspstf-screening-2021],[],"{'origin': 'assistant_generated_synthetic', 's..."
5,CT2-006,dev,screening_age_condition,screening_age_condition,B,eligibility_check,An asymptomatic 51-year-old adult currently sm...,Are the stated age and smoking conditions cons...,已知年龄与当前吸烟暴露符合所引声明的条件；保留共同决策及整体适宜性边界，不把资格判断当成结节类别。,[],answer_with_evidence_and_limits,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",[uspstf-screening-2021],[],"{'origin': 'assistant_generated_synthetic', 's..."
6,CT2-007,dev,screening_exposure_condition,screening_exposure_condition,A,eligibility_check,A healthy asymptomatic 64-year-old current smo...,Does the stated smoking exposure meet the 2021...,10 包年未达到所引 20 包年阈值；年龄满足不代表整个条件满足，不扩大到其他检查指征。,[],answer_with_evidence_and_limits,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",[uspstf-screening-2021],[],"{'origin': 'assistant_generated_synthetic', 's..."
7,CT2-008,dev,screening_exposure_condition,screening_exposure_condition,B,eligibility_check,A healthy asymptomatic 64-year-old current smo...,Do age and exposure meet the stated 2021 USPST...,年龄与已知吸烟条件符合所引声明；这是资格条件核对，不是 Lung-RADS 分类或癌症诊断。,[],answer_with_evidence_and_limits,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",[uspstf-screening-2021],[],"{'origin': 'assistant_generated_synthetic', 's..."
8,CT2-009,dev,screening_quit_duration,screening_quit_duration,A,eligibility_check,An asymptomatic 69-year-old former smoker has ...,How does time since quitting affect applicabil...,16 年已超过所引戒烟时间范围；说明

In [24]:
import numpy as np
evidence_text_list = evidence_df['text'].to_list()

In [19]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs = {"device": "cuda"},
    encode_kwargs = {"batch_size": 8, "normalize_embeddings": True, "prompt": ""},
    query_encode_kwargs = {"batch_size": 1, "normalize_embeddings": True, "prompt": PREFIX}
)

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))"), '(Request ID: 51ab542a-ef20-461d-9c79-edad2acd28da)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


In [28]:
embeddings = np.float32(embedding_model.embed_documents(evidence_text_list))
embeddings.shape

(238, 384)

In [29]:
from retrieve_core import retrieve
from retrieve_evalution import evaluate_retrieval

from functools import partial

retrieve_fn = partial(
    retrieve,
    embedding_model=embedding_model,
    evidences=evidence_df,
    embeddings=embeddings
)

eval_df = evaluate_retrieval(cases_df, retrieve_fn, ks=(1, 3, 5))
eval_df

,case_id,split,group,report,question,gold_evidence_ids,gold_evidence_groups,latency_ms,retrieved_ids,scores,canonical_recall_at_1,group_recall_at_1,canonical_recall_at_3,group_recall_at_3,canonical_recall_at_5,group_recall_at_5
0,CT2-001,dev,screening_framework,Baseline low-dose chest CT within an organized...,"Which reporting framework should I consult, an...",[uspstf-screening-2021::b0020],"[[uspstf-screening-2021::b0020, acr-lungrads-s...",18.837323,"[acr-incidental-2023::b0023, acr-incidental-20...","[0.7901115803905416, 0.7888133218046929, 0.788...",0.0,0.0,0.0,0.0,0.0,0.0
1,CT2-002,dev,screening_framework,Annual screening LDCT. A 67-year-old adult has...,For interpreting this completed screening exam...,[uspstf-screening-2021::b0020],"[[uspstf-screening-2021::b0020, acr-lungrads-s...",3.492110,"[uspstf-screening-2021::b0001, uspstf-screenin...","[0.8288097735517681, 0.8284870891185452, 0.827...",0.0,0.0,0.0,0.0,0.0,0.0
2,CT2-003,dev,missing_examination_purpose,Chest CT: a 63-year-old adult with a single 7 ...,Which guideline pathway applies to this nodule...,"[uspstf-screening-2021::b0020, acr-incidental-...","[[uspstf-screening-2021::b0020], [acr-incident...",3.159230,"[acr-incidental-2023::b0032, acr-incidental-20...","[0.7776420253245564, 0.7742640047264756, 0.773...",0.0,0.0,0.0,0.0,0.0,0.0
3,CT2-004,dev,missing_examination_purpose,A 58-year-old adult has a 9 mm solid nodule on...,Can you select a screening versus incidental-n...,"[uspstf-screening-2021::b0020, acr-incidental-...","[[uspstf-screening-2021::b0020], [acr-incident...",3.131191,"[acr-incidental-2023::b0031, nci-lung-screenin...","[0.7571809063868113, 0.7539529327439111, 0.750...",0.0,0.0,0.0,0.0,0.0,0.0
4,CT2-005,dev,screening_age_condition,An asymptomatic 49-year-old adult currently sm...,Does this person meet the age and smoking cond...,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",3.100770,"[uspstf-screening-2021::b0026, uspstf-screenin...","[0.8418496181653778, 0.829763937543847, 0.8229...",0.0,0.0,1.0,1.0,1.0,1.0
5,CT2-006,dev,screening_age_condition,An asymptomatic 51-year-old adult currently sm...,Are the stated age and smoking conditions cons...,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",3.164063,"[uspstf-screening-2021::b0063, uspstf-screenin...","[0.8121440575890875, 0.8120157518521007, 0.806...",0.0,0.0,1.0,1.0,1.0,1.0
6,CT2-007,dev,screening_exposure_condition,A healthy asymptomatic 64-year-old current smo...,Does the stated smoking exposure meet the 2021...,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",3.212978,"[uspstf-screening-2021::b0063, uspstf-screenin...","[0.8686389649743366, 0.86439777295493, 0.84614...",0.0,0.0,0.0,0.0,0.0,0.0
7,CT2-008,dev,screening_exposure_condition,A healthy asymptomatic 64-year-old current smo...,Do age and exposure meet the stated 2021 USPST...,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",5.135867,"[uspstf-screening-2021::b0026, uspstf-screenin...","[0.8325214193592894, 0.8146176517869488, 0.813...",0.0,0.0,1.0,1.0,1.0,1.0
8,CT2-009,dev,screening_quit_duration,An asymptomatic 69-year-old former smoker has ...,How does time since quitting affect applicabil...,[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",3.122403,"[uspstf-screening-2021::b0063, uspstf-screenin...","[0.8158247810400101, 0.8126320954086543, 0.804...",0.0,0.0,1.0,1.0,1.0,1.0
9,CT2-010,dev,screening_quit_duration,An asymptomatic 69-year-old former smoker has ...,"Do the recorded age, exposure and quit duratio...",[uspstf-screening-2021::b0001],"[[uspstf-screening-2021::b0001, uspstf-screeni...",3.005801,"[uspstf-screening-2021::b0063, uspstf-screenin...","[0.8233527734422424, 0.789803986774983, 0.7741...",0.0,0.0,0.0,1.0,1.0,1.0
